In [2]:
import pandas as pd
import numpy as np
import h5py

path = 'data/ca_his_raw_2019.h5'
with h5py.File(path, "r") as f:
    print(list(f.keys()))
    def show(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(name, obj.shape, obj.dtype)

    f.visititems(show)

['t']
t/axis0 (8600,) |S9
t/axis1 (105120,) int64
t/block0_items (8600,) |S9
t/block0_values (105120, 8600) float64


In [3]:
domain = pd.read_csv('data/ca_meta.csv', dtype={"ID": str})
domain.head()

,ID,Lat,Lng,District,County,Fwy,Lanes,Type,Direction,ID2
0,317802,38.389811,-121.479587,3,Sacramento,I5-N,2,Mainline,N,0
1,312134,38.412564,-121.484319,3,Sacramento,I5-N,2,Mainline,N,1
2,312133,38.428630,-121.487657,3,Sacramento,I5-N,3,Mainline,N,2
3,313159,38.450246,-121.492176,3,Sacramento,I5-N,3,Mainline,N,3
4,319767,38.465539,-121.496450,3,Sacramento,I5-N,3,Mainline,N,4


In [4]:
domain_fwy = ['SR22-E','SR22-W','SR55-N','SR55-S']
domain.loc[domain['Fwy'].isin(domain_fwy), 'ID'].to_string(index=False).split('\n')

['1202590',
 '1214853',
 '1215236',
 '1202595',
 '1214869',
 '1202614',
 '1215092',
 '1202627',
 '1202648',
 '1202691',
 '1214938',
 '1214955',
 '1202705',
 '1202720',
 '1202742',
 '1202753',
 '1214988',
 '1215003',
 '1214805',
 '1202779',
 '1202785',
 '1215252',
 '1214894',
 '1202827',
 '1215017',
 '1202844',
 '1202855',
 '1215043',
 '1202885',
 '1214715',
 '1202901',
 '1212170',
 '1202921',
 '1202949',
 '1202964',
 '1202977',
 '1214881',
 '1215026',
 '1202574',
 '1202564',
 '1214854',
 '1215248',
 '1202599',
 '1214871',
 '1202610',
 '1215091',
 '1202631',
 '1202663',
 '1202676',
 '1214939',
 '1214954',
 '1202701',
 '1202724',
 '1202738',
 '1214972',
 '1214987',
 '1215002',
 '1214806',
 '1202766',
 '1202789',
 '1215250',
 '1202803',
 '1202814',
 '1215018',
 '1202840',
 '1202859',
 '1215044',
 '1202872',
 '1211641',
 '1202912',
 '1202929',
 '1202917',
 '1202938',
 '1202960',
 '1202981',
 '1214882',
 '1202993',
 '1203021',
 '1203035',
 '1203057',
 '1203071',
 '1203082',
 '1210205',
 '12

In [122]:
domain[domain['District'] == 12].drop_duplicates(['Fwy'])

,ID,Lat,Lng,District,County,Fwy,Lanes,Type,Direction,ID2
7647,1204198,33.405160,-117.597992,12,Orange,I5-N,4,Mainline,N,7647
7754,1204193,33.404943,-117.598167,12,Orange,I5-S,4,Mainline,S,7754
7860,1202590,33.774261,-118.034947,12,Orange,SR22-E,3,Mainline,E,7860
7898,1202574,33.774686,-118.038174,12,Orange,SR22-W,3,Mainline,W,7898
7937,1203021,33.651784,-117.908497,12,Orange,SR55-N,4,Mainline,N,7937
7971,1203016,33.651901,-117.908673,12,Orange,SR55-S,3,Mainline,S,7971
8001,1211907,33.784350,-117.879424,12,Orange,SR57-N,5,Mainline,N,8001
8030,1211954,33.777415,-117.874735,12,Orange,SR57-S,2,Mainline,S,8030
8058,1210440,33.545677,-117.674552,12,Orange,SR73-N,3,Mainline,N,8058
8109,1210441,33.545627,-117.674915,12,Orange,SR73-S,3,Mainline,S,8109


In [128]:
domain.loc[domain['District'] == 12, 'Fwy'].value_counts()

Fwy
I5-N       107
I5-S       106
SR241-N     66
SR241-S     64
I405-N      58
I405-S      57
SR73-S      52
SR73-N      51
SR91-W      46
SR91-E      44
SR22-W      39
SR22-E      38
SR55-N      34
SR55-S      30
SR57-N      29
SR57-S      28
SR133-N     24
SR133-S     20
SR261-N     17
SR261-S     16
SR142-E     14
SR142-W      7
I605-N       3
I605-S       3
Name: count, dtype: int64

In [ ]:
def extract_details(fwy, district = 12):
    ids = set(domain.loc[(domain['District'] == district) & domain['Fwy'].isin(fwy), 'ID'])
    with h5py.File(path, "r") as f:
        sensors_raw = [x.decode("utf-8") for x in f["t/axis0"][:]]
        sensors = []
        for i in sensors_raw:
            if i in ids:
                sensors.append(i)
        timestamps = pd.to_datetime(f['t/axis1'][:])

        print('number of sensors:', len(sensors))
        print('first sensors:', sensors[:20])
        print('first timestamps:', timestamps[:5])
        return sensors, len(sensors), timestamps
d12_fwy = domain.loc[domain['District'] == 12, 'Fwy'].unique().tolist() 
domain_sensors, len_of_sensors, timestamps = extract_details(d12_fwy)

number of sensors: 953
first sensors: ['1204198', '1204211', '1204230', '1204244', '1204255', '1210908', '1204268', '1204279', '1213215', '1221232', '1204301', '1204316', '1204328', '1204340', '1204372', '1204384', '1204395', '1204409', '1210926', '1220030']
first timestamps: DatetimeIndex(['2019-01-01 00:00:00', '2019-01-01 00:05:00',
               '2019-01-01 00:10:00', '2019-01-01 00:15:00',
               '2019-01-01 00:20:00'],
              dtype='datetime64[ns]', freq=None)


In [91]:
def get_df(sensors, timestamps, span):
    wanted = pd.Index(sensors)
    with h5py.File(path, "r") as f:
        hdf_sensors = pd.Index(x.decode("utf-8") for x in f["t/axis0"][:])
        positions = hdf_sensors.get_indexer(wanted)

        values = f['t/block0_values'][:, positions]

    df = pd.DataFrame(values, index=timestamps, columns=wanted)
    if span == 1:
        df = df.loc['2019-01-01':'2019-06-30']
    elif span == 2:
        df = df.loc['2019-07-01':'2019-12-31']
    else:
        df = df
    return df.resample('15min').mean()
df = get_df(domain_sensors, timestamps, 3)
df

,1204198,1204211,1204230,1204244,1204255,1210908,1204268,1204279,1213215,1221232,...,1221523,1221550,1221536,1221556,1219551,1202527,1202549,1219560,1202522,1202537
2019-01-01 00:00:00,8.666667,39.333333,57.666667,65.666667,59.000000,33.333333,98.000000,58.333333,57.666667,61.333333,...,101.666667,104.666667,92.666667,76.000000,51.666667,0.000000,NaN,25.666667,64.333333,58.000000
2019-01-01 00:15:00,8.000000,33.666667,73.000000,79.333333,69.333333,39.666667,100.333333,72.000000,73.000000,92.333333,...,177.666667,165.666667,121.333333,98.333333,137.333333,127.333333,NaN,55.666667,122.000000,118.000000
2019-01-01 00:30:00,5.666667,42.000000,74.333333,78.666667,68.333333,34.666667,103.333333,73.333333,74.000000,87.666667,...,255.000000,235.000000,186.333333,161.666667,183.000000,167.333333,NaN,147.333333,148.666667,137.333333
2019-01-01 00:45:00,18.333333,52.666667,79.333333,76.666667,74.333333,35.333333,107.000000,77.000000,82.666667,91.333333,...,270.333333,267.666667,217.333333,179.666667,186.000000,167.666667,NaN,169.000000,156.666667,153.666667
2019-01-01 01:00:00,58.333333,77.666667,89.333333,97.333333,95.666667,84.000000,128.666667,92.000000,96.000000,110.333333,...,280.000000,281.333333,222.000000,180.333333,141.000000,120.333333,NaN,175.333333,171.000000,163.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-31 22:45:00,38.333333,64.666667,68.333333,106.000000,101.333333,52.000000,86.666667,84.666667,86.666667,92.666667,...,260.333333,212.000000,141.000000,144.000000,145.666667,140.000000,135.333333,145.000000,144.000000,132.333333
2019-12-31 23:00:00,55.333333,67.666667,76.333333,116.000000,114.333333,78.333333,95.000000,97.000000,96.333333,105.000000,...,221.333333,191.666667,127.000000,130.000000,130.000000,119.333333,111.000000,47.666667,108.666667,110.333333
2019-12-31 23:15:00,19.000000,51.666667,63.666667,102.333333,102.000000,53.666667,83.000000,79.333333,84.000000,91.333333,...,225.000000,182.000000,115.000000,116.000000,118.000000,115.333333,108.666667,124.000000,125.000000,114.333333
2019-12-31 23:30:00,7.333333,41.666667,45.666667,85.666667,93.000000,0.000000,56.666667,58.666667,59.666667,68.666667,...,196.333333,165.000000,105.333333,107.666667,100.333333,92.333333,86.666667,104.333333,104.666667,94.666667


In [92]:
def sensor_to_fwy(df):
    return domain.set_index("ID").loc[df.columns, "Fwy"]
d12_sensor_to_fwy = sensor_to_fwy(df)
d12_sensor_to_fwy

1204198      I5-N
1204211      I5-N
1204230      I5-N
1204244      I5-N
1204255      I5-N
            ...  
1202527    I605-N
1202549    I605-N
1219560    I605-S
1202522    I605-S
1202537    I605-S
Name: Fwy, Length: 953, dtype: object

In [93]:
def corridor_stats(df, fwy):
    mean = df.fillna(0).T.groupby(fwy).apply(lambda x: x.to_numpy().mean())
    sensor = df.T.groupby(fwy).size()
    std = df.fillna(0).T.groupby(fwy).apply(lambda x: np.std(x.to_numpy()))
    zero_rate = df.fillna(0).T.groupby(fwy).apply(lambda x: (x.to_numpy()==0).mean())

    return pd.concat([sensor, mean, std, zero_rate], axis=1).rename(columns={0:"sensor", 1:"mean", 2: "std", 3:"zero-rate"})

stats = corridor_stats(df, d12_sensor_to_fwy)
stats

,sensor,mean,std,zero-rate
Fwy,,,,
I405-N,58,371.944595,231.385921,0.032001
I405-S,57,381.819050,211.714443,0.005470
I5-N,107,365.932901,201.683295,0.003232
I5-S,106,371.774998,200.145906,0.007419
I605-N,3,239.188659,177.115741,0.155023
I605-S,3,251.391789,130.758981,0.007021
SR133-N,24,89.905826,79.898706,0.013336
SR133-S,20,78.093040,86.831097,0.150308
SR142-E,14,85.677540,90.512900,0.264465


In [129]:
def CSS(df):
    df = df.copy()
    min_max = (df-df.min())/(df.max()-df.min())
    df['CSS'] = 0.25 * (min_max['mean']+min_max['std']+(1-min_max['zero-rate'])+min_max['sensor'])
    return df.sort_values('CSS', ascending=False)

corridor = CSS(stats)
corridor

,sensor,mean,std,zero-rate,CSS
Fwy,,,,,
I5-N,107,365.932901,201.683295,0.003232,0.946157
I5-S,106,371.774998,200.145906,0.007419,0.941947
I405-S,57,381.819050,211.714443,0.005470,0.848698
I405-N,58,371.944595,231.385921,0.032001,0.845150
SR91-E,44,363.624577,171.688119,0.000368,0.755689
SR91-W,46,356.702477,181.646781,0.022546,0.747818
SR57-N,29,361.398246,189.066423,0.002121,0.739585
SR57-S,28,349.400253,183.381676,0.002277,0.720791
SR55-N,34,313.315095,190.238395,0.013433,0.707805


In [99]:
domain_fwy = ['SR22-E','SR22-W','SR55-N','SR55-S']
corridor[corridor.index.isin(domain_fwy)]

,sensor,mean,std,zero-rate,CSS
Fwy,,,,,
SR55-N,34,313.315095,190.238395,0.013433,0.707805
SR22-W,39,292.484772,172.146610,0.009724,0.684151
SR22-E,38,269.838660,161.057517,0.010544,0.649833
SR55-S,30,296.075773,203.279801,0.077861,0.642198


In [130]:
federated_fwy = ['SR261N', 'SR261-S', 'SR133-N', 'SR133-S', 'SR57-N', 'SR57-S']
corridor[corridor.index.isin(federated_fwy)]

,sensor,mean,std,zero-rate,CSS
Fwy,,,,,
SR57-N,29,361.398246,189.066423,0.002121,0.739585
SR57-S,28,349.400253,183.381676,0.002277,0.720791
SR133-N,24,89.905826,79.898706,0.013336,0.375420
SR261-S,16,35.452830,51.180849,0.034926,0.258137
SR133-S,20,78.093040,86.831097,0.150308,0.236891


In [ ]:
def mean_flow(df, options, fwy):
    d = df.fillna(0)
    if options == '24h':
        key = d.groupby(d.index.hour).mean()
    if options == 'week':
        key = d.groupby(d.index.dayofweek).mean()
    mean = key.T.groupby(fwy).mean().T
    return mean

mean_flow_24h = mean_flow(df, '24h', d12_sensor_to_fwy)
domain_fwy = ['SR22-E','SR22-W','SR55-N','SR55-S']
mean_flow_24h_domain = mean_flow_24h[[c for c in domain_fwy if c in mean_flow_24h.columns]]
mean_flow_week = mean_flow(df, 'week', d12_sensor_to_fwy)
mean_flow_week_domain = mean_flow_week[[c for c in domain_fwy if c in mean_flow_week.columns]]

Fwy,SR22-E,SR22-W,SR55-N,SR55-S
0,272.532508,295.053239,312.316167,298.319965
1,280.967052,302.969303,320.447473,305.478099
2,283.620856,306.108755,322.581235,308.090269
3,284.996148,307.755416,322.565245,307.631593
4,296.567074,319.496416,332.479119,320.252682
5,258.426393,281.358662,314.525470,287.550505
6,211.546581,234.449988,268.153794,245.026486


In [100]:
from datetime import datetime, timedelta

def recent_twelve_readings(time, min = 15, freq = 12):
    span = timedelta(minutes = min * freq-1)
    window = df.loc[time - span : time]
    return window

window = recent_twelve_readings(pd.Timestamp("2012-03-10 04:15:00"))
window

,773869,767541,767542,717447,717446,717445,773062,767620,737529,717816,...,772167,769372,774204,769806,717590,717592,717595,772168,718141,769373
2012-03-10 01:30:00,63.375000,66.750000,63.625000,60.625000,68.875000,67.125000,67.125000,65.750000,59.000000,60.375000,...,43.500000,69.000000,0.0,48.375000,61.375000,63.625000,68.625000,61.625000,67.000000,62.125000
2012-03-10 01:45:00,67.625000,66.750000,67.625000,60.750000,64.500000,56.571429,66.250000,64.375000,65.000000,67.750000,...,45.000000,68.750000,0.0,61.000000,69.500000,64.625000,64.250000,62.125000,68.750000,61.125000
2012-03-10 02:00:00,66.000000,66.500000,67.750000,60.000000,66.250000,64.571429,65.625000,66.625000,61.750000,62.250000,...,41.250000,68.571429,0.0,57.625000,69.875000,66.375000,67.750000,60.125000,68.625000,63.285714
2012-03-10 02:15:00,65.000000,65.875000,64.625000,63.500000,66.375000,68.875000,64.875000,65.875000,53.250000,69.428571,...,47.625000,68.500000,0.0,61.250000,62.875000,62.125000,67.500000,62.250000,69.500000,63.125000
2012-03-10 02:30:00,64.222222,67.222222,68.444444,60.333333,67.333333,66.555556,65.444444,66.111111,61.777778,61.666667,...,44.333333,65.666667,0.0,66.111111,55.222222,62.888889,66.444444,63.333333,67.444444,60.888889
2012-03-10 02:45:00,59.666667,66.888889,68.000000,63.111111,67.444444,67.111111,67.555556,66.333333,57.000000,66.444444,...,44.222222,61.333333,0.0,50.250000,62.111111,59.444444,65.888889,60.222222,67.000000,62.333333
2012-03-10 03:00:00,65.500000,65.750000,69.500000,60.875000,68.625000,67.500000,62.875000,61.375000,63.125000,67.375000,...,46.714286,65.250000,0.0,56.750000,63.125000,67.625000,68.125000,65.142857,69.750000,62.875000
2012-03-10 03:15:00,62.875000,62.000000,67.375000,62.625000,68.375000,66.000000,67.000000,63.250000,62.000000,66.857143,...,40.000000,67.000000,0.0,54.375000,66.500000,61.000000,65.375000,68.125000,65.250000,61.250000
2012-03-10 03:30:00,60.375000,65.125000,64.750000,57.000000,66.625000,64.125000,66.375000,67.375000,62.750000,65.500000,...,45.500000,66.625000,0.0,57.750000,63.750000,65.000000,68.875000,65.250000,65.750000,58.375000
2012-03-10 03:45:00,53.250000,68.000000,68.375000,61.125000,64.625000,66.125000,64.750000,63.750000,64.500000,67.500000,...,40.875000,65.125000,0.0,59.000000,66.625000,61.250000,68.000000,64.250000,68.500000,61.750000


In [94]:
row, col = window.shape
row

2

Timestamp('2012-03-10 04:15:00')

In [132]:
from datetime import datetime, timedelta

def short_term_trend(readings, freq = 4):
    subwindow = readings.iloc[-freq:]
    trends = pd.DataFrame(columns= df.columns)
    old_value = subwindow.iloc[0]
    new_value = subwindow.iloc[-1]
    trends = (new_value - old_value)/ (freq-1)
    return subwindow, trends

subwindow, trends = short_term_trend(window)

In [133]:
subwindow

,773869,767541,767542,717447,717446,717445,773062,767620,737529,717816,...,772167,769372,774204,769806,717590,717592,717595,772168,718141,769373
2012-03-10 03:30:00,60.375,65.125,64.750,57.000,66.625,64.125,66.375,67.375,62.75,65.500,...,45.500,66.625,0.0,57.750,63.750,65.000,68.875,65.25,65.75,58.375
2012-03-10 03:45:00,53.250,68.000,68.375,61.125,64.625,66.125,64.750,63.750,64.50,67.500,...,40.875,65.125,0.0,59.000,66.625,61.250,68.000,64.25,68.50,61.750
2012-03-10 04:00:00,62.200,66.200,62.600,61.400,67.800,59.400,63.600,62.400,47.80,68.800,...,39.800,59.200,0.0,60.400,57.000,69.200,68.400,66.40,68.20,58.800
2012-03-10 04:15:00,60.000,66.125,68.000,56.875,63.750,62.000,62.625,68.125,58.25,66.375,...,40.125,64.375,0.0,59.375,60.375,58.625,65.125,65.00,65.00,63.125


In [134]:
trends

773869   -0.125000
767541    0.333333
767542    1.083333
717447   -0.041667
717446   -0.958333
            ...   
717592   -2.125000
717595   -1.250000
772168   -0.083333
718141   -0.250000
769373    1.583333
Length: 207, dtype: float64